# Homework 2 - Vector Search

This notebook follows the Module 2 homework flow.


## Setup


In [1]:
from pathlib import Path
import os

MODULE_DIR = Path("/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/02-vector-search")
os.chdir(MODULE_DIR)

print(Path.cwd())


/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/02-vector-search


In [2]:
import numpy as np
from tqdm.auto import tqdm

from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

embedder = Embedder()


## Q1. Embedding A Query


In [3]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

print(v.shape)
print(v[0])

q1_answer = v[0]


(384,)
-0.020582034180917443


## Load The Data


In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)


72

## Q2. Cosine Similarity


In [5]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = next(doc for doc in documents if doc["filename"] == target_filename)

page_vector = embedder.encode(target_doc["content"])
similarity = page_vector.dot(v)

print(similarity)

q2_answer = similarity


0.36107008472347096


## Q3. Chunking And Search By Hand


In [6]:
chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)


295

In [7]:
texts = [chunk["content"] for chunk in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embedder.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

X.shape


  0%|          | 0/6 [00:00<?, ?it/s]

(295, 384)

In [8]:
scores = X.dot(v)
idx = np.argmax(scores)

best_chunk = chunks[idx]

print(scores[idx])
print(best_chunk["filename"])

q3_answer = best_chunk["filename"]


0.6489016216319834
02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector Search With minsearch


In [9]:
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

q4_results = vindex.search(q4_vector, num_results=5)

for result in q4_results:
    print(result["filename"], result["start"])

q4_answer = q4_results[0]["filename"]


04-evaluation/lessons/05-search-metrics.md 0
04-evaluation/lessons/01-intro.md 0
01-agentic-rag/lessons/05-search.md 0
04-evaluation/lessons/01-intro.md 2000
04-evaluation/lessons/15-next-steps.md 0


## Q5. Text Search vs Vector Search


In [10]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)

vector_results = vindex.search(q5_vector, num_results=5)
text_results = index.search(q5_query, num_results=5)

print("Vector results")
for result in vector_results:
    print(result["filename"], result["start"])

print()
print("Text results")
for result in text_results:
    print(result["filename"], result["start"])

vector_files = {result["filename"] for result in vector_results}
text_files = {result["filename"] for result in text_results}

q5_answer = vector_files - text_files
q5_answer


Vector results
02-vector-search/lessons/08-pgvector.md 0
02-vector-search/lessons/08-pgvector.md 3000
03-orchestration/lessons/05-rag.md 2000
02-vector-search/lessons/08-pgvector.md 1000
02-vector-search/lessons/08-pgvector.md 2000

Text results
02-vector-search/lessons/02-embeddings.md 4000
03-orchestration/lessons/05-rag.md 1000
02-vector-search/lessons/01-intro.md 0
03-orchestration/lessons/05-rag.md 0
02-vector-search/lessons/01-intro.md 1000


{'02-vector-search/lessons/08-pgvector.md'}

## Q6. Hybrid Search


In [11]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


In [12]:
q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)

vector_results = vindex.search(q6_vector, num_results=5)
text_results = index.search(q6_query, num_results=5)

results = rrf([vector_results, text_results])

for result in results:
    print(result["filename"], result["start"])

q6_answer = results[0]["filename"]
q6_answer


01-agentic-rag/lessons/13-function-calling.md 4000
01-agentic-rag/lessons/01-intro.md 2000
01-agentic-rag/lessons/14-agentic-loop.md 0
04-evaluation/lessons/02-ground-truth.md 1000
01-agentic-rag/lessons/16-other-frameworks.md 0


'01-agentic-rag/lessons/13-function-calling.md'